# Arkenstone ARK-020 V2 — Multi-Skill Continual Cognition (repaired)

Cell 0 prints a READ-ONLY resume scan with a SAFE ACTION. Do not skip it.
The campaign is exact-resumable: if a session dies, rerun cell 2 on a fresh T4.

In [ ]:
import json, subprocess, sys, os
PINNED_RUNNER_COMMIT = '3814d26ae9e3227f0c3d8a2742915f6875598fc6'
REPO = '/content/An-Ra-the-new-AGI-ark020v2'
if not os.path.exists(REPO):
    subprocess.run(['git','clone','--depth','50','--branch','Arkenstone','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',REPO], check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',PINNED_RUNNER_COMMIT], check=True)
head = subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip()
assert head == PINNED_RUNNER_COMMIT, (head, PINNED_RUNNER_COMMIT)
print('PINNED COMMIT OK:', head[:12])
r = subprocess.run([sys.executable, os.path.join(REPO,'experiments/ARK-020-V2/run_ark020_v2.py'), '--mode', 'scan'])
print('SAFE ACTION above. If it says STOP - DO NOT RUN, do not proceed to cell 2.')

In [ ]:
import subprocess, sys, torch, os
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU before running.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
paths = [
 'experiments/ARK-020-V2/ark020_v2_core.py',
 'experiments/ARK-020-V2/run_ark020_v2.py',
 'tests/test_ark020_v2.py',
 'experiments/ARK-019/ark019_v4_core.py',
 'experiments/ARK-019/run_ark019_v4.py',
 'experiments/ARK-019/run_ark019_v3.py',
 'experiments/ARK-018/ark018_v3_common.py',
 'experiments/ARK-018/ark018_v3_binding_fast.py',
]
for p in paths:
    subprocess.run([sys.executable,'-m','py_compile',os.path.join(REPO,p)], check=True)
print('compile gate: PASS on', len(paths), 'files')
r = subprocess.run([sys.executable,'-m','unittest','tests.test_ark020_v2'], cwd=REPO)
assert r.returncode == 0, 'test suite failed (pure + integration contracts + CPU smoke)'
print('ALL TEST LAYERS: PASS (pure, contracts, validity, checkpoint, fail-closed, golden, CPU smoke)')

In [ ]:
# Full campaign. Exact-resumable: rerun this cell on a fresh T4 to continue.
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
runner = os.path.join(REPO,'experiments/ARK-020-V2/run_ark020_v2.py')
env = dict(os.environ); env['PYTHONUNBUFFERED']='1'
proc = subprocess.run([sys.executable, runner, '--mode', 'all'], cwd=REPO, env=env)
print('CAMPAIGN RETURN CODE:', proc.returncode)
if proc.returncode != 0:
    print('Partial zip written. Rerun this cell to resume exactly; check cell 0 first.')

In [ ]:
from pathlib import Path
root = Path('/content/drive/MyDrive/genisis-arkenstone/ARK020_V2_CONTINUAL')
res = root / 'ARK-020_V2_RESULT.json'
if res.exists():
    print('ARK-020 V2 RESULT:')
    print(res.read_text())
z = root / 'ARKENSTONE_ARK020_V2_CONTINUAL_RESULTS.zip'
if not z.exists():
    z = root / 'ARKENSTONE_ARK020_V2_CONTINUAL_PARTIAL.zip'
if z.exists():
    print('ZIP:', z, z.stat().st_size, 'bytes')
    try:
        from google.colab import files
        files.download(str(z))
    except Exception as exc:
        print('Manual download from Drive: genisis-arkenstone/ARK020_V2_CONTINUAL/', exc)
else:
    print('No bundle yet - campaign still incomplete; rerun cell 2.')